In [14]:
import weaviate
import os
from weaviate.classes.init import Auth
from langchain_weaviate import WeaviateVectorStore
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv

In [15]:
load_dotenv()

True

In [16]:
documents = [
Document(page_content="Python is a programming language."),
Document(page_content="Machine learning helps computers learn from data."),
Document(page_content="Deep learning uses neural networks."),
Document(page_content="Weaviate is a vector database."),
Document(page_content="Semantic search finds information based on meaning.")
]

In [17]:
embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", output_dimensionality=100)

In [48]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"])
)

In [47]:
client.is_ready()

The `WeaviateClient` is closed. Run `client.connect()` to (re)connect!


False

In [44]:
import os
import weaviate
from weaviate.classes.init import Auth
import weaviate.classes.config as wvc
from langchain_weaviate import WeaviateVectorStore

# 1. Establish the connection
with weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"])
) as client:
    
    # 2. Hard delete the broken collection if it exists
    if client.collections.exists("LangChainDemo"):
        client.collections.delete("LangChainDemo")
        print("Deleted old non-vector collection.")
        
    # 3. Explicitly create it with the HFRESH Vector Index configuration
    client.collections.create(
        name="LangChainDemo",
        description="Collection for LangChain demo with vector indexing enabled",
        vector_index_config=wvc.Configure.VectorIndex.hfresh(), # 👈 CHANGED TO HFRESH
    )
    print("Collection explicitly recreated with a valid hfresh vector index!")

    # 4. Ingest the documents into the configured collection
    vector_store = WeaviateVectorStore.from_documents(
        documents=documents,
        embedding=embedding_model,
        client=client,
        index_name="LangChainDemo"
    )
    
    print("All documents successfully embedded and ingested into the hfresh index!")


c:\NJ\Projects\LangChain\genai_venv\Lib\site-packages\weaviate\warnings.py:355: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
C:\Users\njnjp\AppData\Local\Temp\ipykernel_17300\4274499987.py:11: ResourceWarning: unclosed <ssl.SSLSocket fd=1932, family=2, type=1, proto=0, laddr=('192.168.1.23', 55752), raddr=('3.75.162.244', 443)>
  ) as client:


Deleted old non-vector collection.


c:\NJ\Projects\LangChain\genai_venv\Lib\site-packages\weaviate\warnings.py:206: DeprecationWarning: Dep025: You are using the `vector_index_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead defining `vector_index_config` as a sub-argument.
            
  warnings.warn(


Collection explicitly recreated with a valid hfresh vector index!
All documents successfully embedded and ingested into the hfresh index!


In [51]:
vector_store._client.connect()

query = "how do computers learn?"

results = vector_store.similarity_search(
    query,
    k=2
)

In [52]:
results

[Document(metadata={}, page_content='Machine learning helps computers learn from data.'),
 Document(metadata={}, page_content='Deep learning uses neural networks.')]

In [53]:
client.close()